# Revisão humana, aprovação e rollback

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="07_registry_governanca"


1. Modelos/versões e pedidos de revisão. Registro é candidato; aprovação exige token separado de operador e versão explícita.

In [ ]:
import mlflow
mlflow.set_tracking_uri(cfg.tracking);client=mlflow.MlflowClient()
from urllib.parse import urlencode
catalog=call(cfg,"GET","/v1/governance/models?"+urlencode(dict(prefix=params.registry_prefix,page_token=params.registry_page_token,limit=100)))
display(pd.DataFrame(catalog["items"]));display({"next_page_token":catalog["next_page_token"]})
versions=client.search_model_versions("name='"+params.model_name+"'",max_results=100)
display(pd.DataFrame([dict(model=v.name,version=v.version,run_id=v.run_id,tags=v.tags) for v in versions]))
display(pd.DataFrame(call(cfg,"GET","/v1/governance/requests")))


2. Inspecionar artefatos e proveniência da versão selecionada. Esperado: hashes iguais; nenhuma aprovação implícita.

In [ ]:
if params.model_version:
    from src.operations.training import trace_for
    from src.workbench.verification import verify_model_storage
    trace=trace_for(client,client.get_model_version(params.model_name,params.model_version))
    evidence["candidate"]=trace;evidence["storage"]=verify_model_storage(trace)
    display(trace);display(pd.DataFrame(evidence["storage"]))


3. Ação opt-in: solicitar revisão ou decidir pedido identificado. APPROVE/REVOKE/ROLLBACK requerem justificativa; reviewer vem da credencial, não do texto digitado.

In [ ]:
if params.governance_action!="read":
    params.authorize()
    if params.governance_action=="request":
        if not params.model_version or len(params.reason)<8: raise ValueError("Versão e justificativa obrigatórias")
        evidence["review"]=call(cfg,"POST","/v1/governance/requests",dict(model_name=params.model_name,model_version=params.model_version,
            action=params.review_action,reason=params.reason,requested_by=params.requester))
    else:
        if not params.approval_id: raise ValueError("approval_id obrigatório")
        evidence["review"]=call(cfg,"POST",f"/v1/governance/requests/{params.approval_id}/review",
            dict(approve=params.governance_action=="approve",confirm="REVIEW_EXPLICIT_VERSION"),operator=True)
    display(evidence["review"])
else:
    evidence["review"]="NOT_EXECUTED: leitura"


Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [ ]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
display(report(NOTEBOOK_ID,evidence,status=report_status,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))
